# EURES thesis pilot: reproducible analysis

This notebook rebuilds the analytical dataset for every EURES country sample, runs all descriptive, inferential and predictive models, and displays the results for one selected country.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in roots if (p / 'build_analysis.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory.')
%cd $PROJECT_ROOT
print(f'Project root: {PROJECT_ROOT}')

## 1. Rebuild the analytical dataset (every country)

In [ ]:
%run build_analysis.py

## 2. Run descriptive analysis, regressions and robustness checks (every country)

In [ ]:
%run analyze.py

## 3. Select a country to display

Change `COUNTRY_CODE` to any two-letter code with a `reports/<code>/` folder (e.g. `fr`, `de`, `nl`, `cs`).

In [ ]:
COUNTRY_CODE = 'fr'

report_dir = Path('reports') / COUNTRY_CODE
if not report_dir.exists():
    raise FileNotFoundError(
        f'No reports for "{COUNTRY_CODE}". Available: '
        f'{sorted(p.name for p in Path("reports").iterdir())}'
    )
print(f'Displaying: {report_dir}')

## 4. Data quality summary

In [ ]:
quality = pd.read_csv(report_dir / 'quality_summary.csv')
display(quality)

## 5. Core descriptive results

In [ ]:
salary_mix = pd.read_csv(report_dir / 'salary_by_skill_mix.csv')
display(salary_mix.round(1))
display(Image(filename=report_dir / 'salary_overview.png'))

## 6. Adjusted regression results

In [ ]:
coefs = pd.read_csv(report_dir / 'regression_coefficients.csv')
main_mix = coefs[coefs['model'].eq('adjusted_trimmed')]
categories = coefs[coefs['model'].eq('category_adjusted_trimmed')]
display(main_mix[['term', 'percent_association', 'percent_ci_low', 'percent_ci_high', 'p_value']].round(3))
display(categories[['term', 'percent_association', 'percent_ci_low', 'percent_ci_high', 'p_value']].round(3))
display(Image(filename=report_dir / 'adjusted_coefficients.png'))

## 7. Direct technical + soft comparison

In [ ]:
contrast = pd.read_csv(report_dir / 'skill_mix_contrast.csv')
display(contrast.round(3))

## 8. Predictive performance: random and temporal hold-outs

In [ ]:
metrics = pd.read_csv(report_dir / 'prediction_metrics.csv')
display(metrics.round(3))